# 23 · The told model as the browser runs it

Notebook 22's model, told the slice QP, the loop filter and I or P, exported for the client (desktop-streamer,
`stream-enhance.js`). Every frame the client reads three numbers off the slice headers (`stream-h264.js`) and hands them
to the graph beside the picture:

    input      [N, 3, H, W]   float32 RGB in [0, 1], a batch of one frame's tiles
    condition  [3]            slice QP / 51, loop filter (0 or 1), I frame (0 or 1)
    output     [N, 3, 2H, 2W]

**The condition never meets a pixel.** In `ToldLuma` each condition is a 1×1 shift added after a convolution, which is
the same as that convolution's bias moving with the frame: `b + W c`. So the graph computes the three biases first - a
1×1 convolution over a 1×1 picture holding the condition, 24 numbers each - and hands them to the blind model's
convolutions as their bias input. The per-pixel work is the blind graph's exactly; the told part is three tiny
convolutions and four `Reshape`s on 3 and 24 numbers.

Three files:

- `dist/upscale/upscale_qp_slice_24x2_tile320x180.onnx` - one 328×188 tile, static, for the benchmark page (which now
  feeds every input the shape the graph states, so `condition` gets three numbers).
- `dist/upscale/client/upscale.onnx` - `N`, `H` and `W` symbolic, the client's contract; it replaces the mock in
  desktop-streamer's `src/client/web/media/models/`.
- `dist/upscale/client/upscale_fp16.onnx` - the same graph with float16 weights and arithmetic, float32 at both ends, which
  the client runs instead on a GPU with `shader-f16` (see 6). It goes beside `upscale.onnx` in `media/models/`.

Needs: notebook 22 (`checkpoints/upscale_qp_slice.pt`) and notebook 21's validation pairs.

In [ ]:
"""Settings."""

import copy
import os
import tempfile
import time
from pathlib import Path

import numpy as np
import onnx
import onnxruntime as ort
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

os.environ["DATASETS_DIR"] = "/mnt/d/datasets"   # where the datasets are: set before helpers is imported

from helpers import models, patches, sources, training, webexport

CHECKPOINT = training.CHECKPOINT_DIR / "upscale_qp_slice.pt"                # notebook 22
PAIRS_DIR = sources.ROOT / "upscale" / "qp_slice_pairs"                     # notebook 21
CLIENT_DIR = webexport.MODELS_DIR / "client"                                # what the client's media/models/ takes
NAME = "upscale_qp_slice_24x2"
STEP = (180, 320)                                                           # the client's tile step (height, width)
OPSET = 17

saved = training.load_checkpoint(CHECKPOINT)
told = models.ToldLuma(scale=saved["scale"], **saved["arguments"])
told.load_state_dict(saved["state_dict"])
told = told.cpu().eval()
CONDITIONS, QP_MAX, HALO, SCALE = saved["told"], saved["qp_max"], saved["halo"], saved["scale"]
TILE = (STEP[0] + 2 * HALO, STEP[1] + 2 * HALO)
print("told", CONDITIONS, "qp /", QP_MAX, "halo", HALO, "tile", TILE, "seed", saved["seed"], "epoch", saved["best_epoch"])

## 1. The graph

`Shipped` is the trained model rewritten the way the graph will hold it, checked against the trained one before
anything is exported.

In [ ]:
class Shipped(nn.Module):
    """`ToldLuma` as the client runs it: the picture and three numbers in, the picture out.

    The condition is one frame's, so it never meets a pixel: each shift is a 1x1 convolution over a 1x1 picture
    that holds the condition, with the bias of the convolution it shifts as its own bias, and what comes out is
    that convolution's bias for this frame. The big convolutions are the blind model's, with a bias that is an
    input instead of a weight - nothing per pixel is added."""

    def __init__(self, told):
        super().__init__()
        told = copy.deepcopy(told)
        self.skip, self.shuffle, self.tail = told.skip, told.shuffle, told.tail
        self.stem, self.act, self.blocks = told.body[0], told.body[1], told.body[2:]
        convs = [self.stem, *[block.conv for block in self.blocks]]
        self.biases = nn.ModuleList()
        for conv, shift in zip(convs, told.shifts):
            line = nn.Conv2d(shift.in_channels, shift.out_channels, 1)
            with torch.no_grad():
                line.weight.copy_(shift.weight)
                line.bias.copy_(conv.bias)
            self.biases.append(line)

    def forward(self, input, condition):
        c = condition.reshape(1, -1, 1, 1)
        bias = [line(c).reshape(-1) for line in self.biases]
        h = self.act(F.conv2d(input, self.stem.weight, bias[0], padding=self.stem.padding))
        for block, b in zip(self.blocks, bias[1:]):
            h = h + block.point(block.act(F.conv2d(h, block.conv.weight, b, padding=block.conv.padding)))
        luma = self.tail(h)
        return self.shuffle(self.skip(input) + torch.cat([luma, luma, luma], dim=1))


def condition_of(qp, deblocked, intra):
    """What the client feeds: the slice QP / 51, the loop filter, I frame."""
    return np.array([qp / QP_MAX, float(deblocked), float(intra)], dtype=np.float32)


shipped = Shipped(told).eval()
probe = torch.rand(3, 3, *TILE)
with torch.no_grad():
    for frame in ((22, 1, 0), (38, 0, 1), (51, 1, 1)):
        c = torch.from_numpy(condition_of(*frame))
        difference = (shipped(probe, c) - told(probe, c.reshape(1, -1, 1, 1).expand(3, -1, 1, 1))).abs().max().item()
        assert difference < 1e-5, difference
print("the rewritten model computes the trained one")

## 2. Export

The classic exporter, as `webexport.export` uses, with two inputs. The fields a reader of the file needs to feed it are
in its `metadata_props`.

In [ ]:
def export(model, path, size, dynamic):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.onnx.export(
        copy.deepcopy(model).eval(), (torch.rand(1, 3, *size), torch.from_numpy(condition_of(30, 1, 0))), str(path),
        opset_version=OPSET, input_names=["input", "condition"], output_names=["output"],
        dynamic_axes=({"input": {0: "N", 2: "H", 3: "W"}, "output": {0: "N", 2: "H2", 3: "W2"}} if dynamic else None),
        dynamo=False)
    graph = onnx.load(str(path))
    onnx.checker.check_model(graph)
    return path


told_fields = {"told": ",".join(CONDITIONS), "qp_max": QP_MAX, "qp": "slice header",
               "condition": "slice QP / 51, loop filter 0 or 1, I frame 0 or 1"}

tile_path = export(shipped, webexport.MODELS_DIR / f"{NAME}_tile{STEP[1]}x{STEP[0]}.onnx", TILE, dynamic=False)
webexport.write_metadata(tile_path, {"label": "told QP + frame 24x2 (notebook 23)", "step": STEP[1], "step_h": STEP[0],
                                     "halo": HALO, "batch": 1, **told_fields})
client_path = export(shipped, CLIENT_DIR / "upscale.onnx", TILE, dynamic=True)
webexport.write_metadata(client_path, {"label": "told QP + frame 24x2 (notebook 23)", "halo": HALO, "scale": SCALE,
                                       **told_fields})
# float32 at both ends: the client's tiles are float32 on the GPU already, and a Cast at each end is cheaper than
# teaching its two shaders float16
half_path = webexport.to_float16(webexport.describe(client_path), "upscale_fp16.onnx", models_dir=CLIENT_DIR,
                                 keep_io_types=True)["path"]
for path in (tile_path, client_path, half_path):
    print(webexport.summary_line(webexport.describe(path)))
    graph = onnx.load(str(path)).graph
    for value in [*graph.input, *graph.output]:
        dims = [d.dim_value if d.HasField("dim_value") else d.dim_param for d in value.type.tensor_type.shape.dim]
        print("   ", value.name, dims)

## 3. Check

Both files against the model baked for one frame (`ToldLuma.baked`, which notebook 19 proved computes the told model),
at six conditions, at the client's batch of tiles and at sizes it never uses. Then the client graph with its symbolic
dimensions written as numbers, which is what the client's WebGL path does to it before making a session. Last, the
float16 graph against the float32 one: it has to stay within an 8-bit step of it.

In [ ]:
def runner(path):
    return webexport.session(path)


def run(session, x, condition):
    return session.run(None, {"input": x.astype(np.float32), "condition": condition})[0]


tile_session, client_session = runner(tile_path), runner(client_path)
worst = 0.0
for frame in ((0, 0, 0), (18, 1, 0), (30, 0, 0), (34, 1, 1), (42, 0, 1), (51, 1, 0)):
    condition = condition_of(*frame)
    x = np.random.rand(36, 3, *TILE).astype(np.float32)
    with torch.no_grad():
        expected = told.baked(condition)(torch.from_numpy(x)).numpy()
    worst = max(worst, float(np.abs(run(client_session, x, condition) - expected).max()))
    worst = max(worst, float(np.abs(run(tile_session, x[:1], condition) - expected[:1]).max()))
    for size in ((1, 3, 64, 48), (2, 3, 720 // 2 + 8, 1280 // 2 + 8)):   # any size the client hands it
        y = np.random.rand(*size).astype(np.float32)
        with torch.no_grad():
            worst = max(worst, float(np.abs(run(client_session, y, condition) - told.baked(condition)(torch.from_numpy(y)).numpy()).max()))
print(f"both graphs compute the baked model for every condition: largest difference {worst:.1e}")
assert worst < 1e-4

# the client's WebGL path writes the symbolic dimensions as the batch's before it makes a session
fixed = onnx.load(str(client_path))
for value in [fixed.graph.input[0], fixed.graph.output[0]]:
    for dim, size in zip(value.type.tensor_type.shape.dim, (36, 3, TILE[0] * (SCALE if value.name == "output" else 1),
                                                            TILE[1] * (SCALE if value.name == "output" else 1))):
        dim.dim_value = size
onnx.checker.check_model(fixed)
fixed_session = ort.InferenceSession(fixed.SerializeToString(), providers=["CPUExecutionProvider"])
x = np.random.rand(36, 3, *TILE).astype(np.float32)
c = condition_of(33, 1, 0)
assert np.abs(fixed_session.run(None, {"input": x, "condition": c})[0] - run(client_session, x, c)).max() < 1e-6
print("with its dimensions fixed, as on WebGL, it computes the same")

# the float16 graph, at the client's batch and every condition, against the float32 one
half_session = runner(half_path)
worst = 0.0
for frame in ((0, 0, 0), (18, 1, 0), (30, 0, 0), (34, 1, 1), (42, 0, 1), (51, 1, 0)):
    condition = condition_of(*frame)
    x = np.random.rand(16, 3, *TILE).astype(np.float32)
    worst = max(worst, float(np.abs(run(half_session, x, condition) - run(client_session, x, condition)).max()))
print(f"the float16 graph is within {worst * 255:.2f}/255 of the float32 one")
assert worst < 1 / 255

## 4. On the validation patches

Notebook 21's validation patches through the client graph in onnxruntime, each with its own frame's condition, scored
against the trained model in torch. The two columns must agree; they also give the trained model's numbers alone (notebook
22's table is the mean of two seeds).

In [ ]:
manifest = training.load_manifest(PAIRS_DIR)
ENCODERS = manifest["encoders"]
trim = training.trimmer(manifest)


class WithConditions(Dataset):
    """Notebook 22's: a split's pairs as ((LR, conditions (3, 1, 1)), HR), and each patch's labels."""

    def __init__(self, split):
        self.pairs = patches.PatchPairs(PAIRS_DIR, split, manifest, augment=False)
        files = manifest["splits"][split]
        self.labels = {name: np.load(PAIRS_DIR / files[name]) for name in ("qp", "deblocked", "intra", "encoder")}
        values = np.stack([condition_of(*row) for row in zip(self.labels["qp"], self.labels["deblocked"], self.labels["intra"])])
        self.conditions = torch.from_numpy(values)[:, :, None, None]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        lr, hr = self.pairs[index]
        return (lr, self.conditions[index]), hr

    def where(self, encoder=None, intra=None):
        keep = np.ones(len(self), dtype=bool)
        if encoder is not None:
            keep &= self.labels["encoder"] == ENCODERS.index(encoder)
        if intra is not None:
            keep &= self.labels["intra"] == intra
        return np.flatnonzero(keep)


class Trained(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model.to(training.DEVICE).eval()

    def forward(self, x, condition):
        return self.model(x, condition)


class Graph(nn.Module):
    """The client's graph, run by onnxruntime: the patches of a batch that share a frame's condition go in as one batch,
    as a frame's tiles do."""

    def __init__(self, session):
        super().__init__()
        self.session = session

    def forward(self, x, condition):
        x, condition = x.cpu().numpy(), condition.flatten(1).cpu().numpy()
        out = np.empty((x.shape[0], 3, x.shape[2] * SCALE, x.shape[3] * SCALE), dtype=np.float32)
        frames, which = np.unique(condition, axis=0, return_inverse=True)
        for index, frame in enumerate(frames):
            rows = np.flatnonzero(which.ravel() == index)
            out[rows] = run(self.session, x[rows], frame)
        return torch.from_numpy(out).to(training.DEVICE)


class Bicubic(nn.Module):
    def forward(self, x, condition=None):
        return models.bicubic(x, SCALE)


val_set = WithConditions("val")
GROUPS = [("all", {}), *[(f"{e} {kind}", {"encoder": e, "intra": kind == "I"}) for e in ENCODERS for kind in "PI"]]
print(f"{'':<12}{'patches':>8}{'bicubic':>9}{'trained':>18}{'graph':>18}{'difference':>12}")
for label, where in GROUPS:
    indices = val_set.where(**where)
    data = DataLoader(Subset(val_set, indices), batch_size=128)
    base, trained, graph = (training.evaluate(model, data, trim) for model in (Bicubic(), Trained(copy.deepcopy(told)), Graph(client_session)))
    cells = [f"{s['ssim'] - base['ssim']:+.4f} / {s['y'] - base['y']:+.2f}" for s in (trained, graph)]
    print(f"{label:<12}{len(indices):8}{base['ssim']:9.4f}{cells[0]:>18}{cells[1]:>18}{abs(graph['ssim'] - trained['ssim']):12.1e}")
    assert abs(graph["ssim"] - trained["ssim"]) < 1e-4

## 5. What the condition costs

The client graph against the blind graph exported the same way, on a 960×540 frame's 9 tiles, at onnxruntime's three
optimization levels on the desktop CPU. The browser runs neither of these providers, so this is for whether the condition
adds work, not for how long a frame takes there - that is the benchmark page's to measure.

In [ ]:
class Baked(nn.Module):
    """The blind model's graph, exported the same way: one frame's biases as weights, no condition input."""

    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input):
        return self.model(input)


baked_path = Path(tempfile.mkdtemp()) / "baked.onnx"
torch.onnx.export(Baked(told.baked(condition_of(30, 1, 0))).eval(), (torch.rand(1, 3, *TILE),), str(baked_path),
                  opset_version=OPSET, input_names=["input"], output_names=["output"],
                  dynamic_axes={"input": {0: "N", 2: "H", 3: "W"}, "output": {0: "N", 2: "H2", 3: "W2"}}, dynamo=False)
frame = np.random.rand(9, 3, *TILE).astype(np.float32)   # a 960x540 stream drawn at 1920x1080: 3x3 tiles
c = condition_of(30, 1, 0)


def ms_per_run(path, feeds, level, seconds=2.0):
    options = ort.SessionOptions()
    options.graph_optimization_level = level
    session = ort.InferenceSession(str(path), options, providers=["CPUExecutionProvider"])
    for _ in range(2):
        session.run(None, feeds)
    runs, started = 0, time.perf_counter()
    while time.perf_counter() - started < seconds or runs < 3:
        session.run(None, feeds)
        runs += 1
    return (time.perf_counter() - started) / runs * 1000


LEVELS = {"basic": ort.GraphOptimizationLevel.ORT_ENABLE_BASIC, "extended": ort.GraphOptimizationLevel.ORT_ENABLE_EXTENDED,
          "all": ort.GraphOptimizationLevel.ORT_ENABLE_ALL}
print(f"9 tiles of {TILE[1]}x{TILE[0]} (a 960x540 frame) on the desktop CPU, ms a run")
print(f"{'optimization':<14}{'told':>8}{'blind':>8}")
for name, level in LEVELS.items():
    told_ms = ms_per_run(client_path, {"input": frame, "condition": c}, level)
    blind_ms = ms_per_run(baked_path, {"input": frame}, level)
    print(f"{name:<14}{told_ms:8.1f}{blind_ms:8.1f}")

## 6. What it says

- **The graph computes the trained model.** About 1e-6 at most from the model baked for each frame, at every size; on the 9,628
  validation patches the same SSIM-Y to 1.5e-8: **+0.0243 / +1.52 dB over bicubic** (notebook 8's graph, which ships
  today, is +0.0141 / +0.95 on the same patches). x264 P +0.0302, NVENC P +0.0186, x264 I +0.0288, NVENC I +0.0171.
- **The condition adds no work of its own.** At the basic and extended levels the told and the blind graph take the same
  time to the millisecond. At "all" the blind one takes 56-58 ms against the told one's 84: that level adds the CPU provider's NCHWc layout,
  which only rewrites a convolution whose bias is a constant, so the told graph keeps the plain kernels and pays the
  layout changes around them. That pass belongs to the CPU provider alone; WebGPU and WebGL never run it. The
  benchmark page can now run the tile graph, and that is the measurement for the browser.
- **The client feeds three numbers a frame,** as a CPU tensor beside the picture's GPU buffer, from what `stream-h264.js`
  already returns: `qp / 51`, `isDeblocked`, `type === "I"`. A frame the parser could not read gets the last frame's.
- `Reshape` is outside `BUDGET`, but only ever on 3 and 24 numbers; it moves nothing.
- **The float16 graph is what a GPU with `shader-f16` should run.** It is within 0.17/255 of the float32 one. In the client
  on a GTX 1660 Ti (`bottlenecks_gtx1660ti.md`) it takes a 1280×720 stream's frame from 65.5 to 41 ms and a 1080p one from
  215 to 86 ms. At 720p, about 15 ms of that is the convolutions, which are about 30% faster. The other ~10 ms is ONNX
  Runtime Web's buffer cache. With the float32 graph at 16 tiles it destroyed and made a 128 MiB buffer on every frame,
  where the 94.7 MB activations fall. With the float16 graph it makes none after the first frame at any size up to 1080p.